In [ ]:
# This cell mounts Google Drive so the notebook can access the teacher-generated training data and save the new Gemma QLoRA experiment.

from google.colab import drive

drive.mount("/content/drive")

print("✅ Google Drive mounted successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully.


In [ ]:
# This cell installs the stable Transformers/PEFT/QLoRA package versions needed for Gemma 3 1B training in the new Colab notebook.

%pip install -q --no-cache-dir \
    "transformers==4.56.2" \
    "bitsandbytes>=0.46.1" \
    "accelerate>=1.10.0" \
    "peft>=0.20.0" \
    "datasets>=3.0.0" \
    "sentencepiece" \
    "torchao>=0.17.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 97.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 168.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 201.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 249.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 157.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 217.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
# This cell repairs the Hugging Face Hub installation so it is compatible with Transformers 4.56.2 and removes the XetAuthorizationError import mismatch.

%pip uninstall -y huggingface-hub hf-xet
%pip install -q --no-cache-dir --force-reinstall "huggingface-hub==0.36.2"

Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
Found existing installation: hf-xet 1.5.1
Uninstalling hf-xet-1.5.1:
  Successfully uninstalled hf-xet-1.5.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 212.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.6/206.6 kB 287.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 172.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 202.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 307.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 261.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 199.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# This cell verifies that the GPU and required QLoRA training libraries are available with the expected versions before we continue.

import torch
import transformers
import bitsandbytes
import peft
import accelerate
import huggingface_hub

print("=" * 80)
print("ENVIRONMENT CHECK")
print("=" * 80)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
        "GiB"
    )

print("\nPyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

print("\n✅ Environment check complete.")

ENVIRONMENT CHECK
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GiB

PyTorch: 2.11.0+cu128
Transformers: 4.56.2
bitsandbytes: 0.50.0
PEFT: 0.20.0
Accelerate: 1.14.0
Hugging Face Hub: 0.36.2

✅ Environment check complete.


In [ ]:
# This cell logs into Hugging Face so the notebook can access the gated google/gemma-3-1b-it model.

from huggingface_hub import notebook_login, whoami

notebook_login()

account = whoami()

print("\n✅ Hugging Face login successful.")
print("Logged in as:", account["name"])


✅ Hugging Face login successful.
Logged in as: Poojitha1997


In [ ]:
# This cell verifies that the fixed teacher-generated GSM8K train and validation JSONL files exist and counts the number of examples in each.

import os
import json

TRAIN_FILE = (
    "/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/"
    "data_final_v3/"
    "teacher_gsm8k_train_qwen3_14b_awq_gsm8k_teacher_v4_434a9551e7_full_sft.jsonl"
)

VAL_FILE = (
    "/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma/"
    "data_final_v3/"
    "teacher_gsm8k_val_qwen3_14b_awq_gsm8k_teacher_v4_434a9551e7_full_sft.jsonl"
)

if not os.path.isfile(TRAIN_FILE):
    raise FileNotFoundError(TRAIN_FILE)

if not os.path.isfile(VAL_FILE):
    raise FileNotFoundError(VAL_FILE)

def count_jsonl(path):
    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                json.loads(line)
                count += 1
    return count

train_count = count_jsonl(TRAIN_FILE)
val_count = count_jsonl(VAL_FILE)

print("=" * 80)
print("TRAINING DATA CHECK")
print("=" * 80)
print("Train file exists:", os.path.isfile(TRAIN_FILE))
print("Validation file exists:", os.path.isfile(VAL_FILE))
print()
print("Train examples:", train_count)
print("Validation examples:", val_count)

if train_count != 1922:
    raise ValueError(f"Expected 1922 train examples, found {train_count}")

if val_count != 485:
    raise ValueError(f"Expected 485 validation examples, found {val_count}")

print("\n✅ Training and validation data are ready.")

TRAINING DATA CHECK
Train file exists: True
Validation file exists: True

Train examples: 1922
Validation examples: 485

✅ Training and validation data are ready.


In [ ]:
# This cell defines the experiment configuration. Every hyperparameter used below is read from here, so a new experiment only requires changing EXPERIMENT_NAME (and whichever values are being tested) rather than editing multiple cells.

# Change this for a new experiment. Each name gets its own Drive folder, so
# a new run never overwrites an earlier one ("v1" is the run that produced
# gemma3_1b_qlora_v1/final_best_adapter — learning rate 2e-4, 3 epochs —
# the adapter loaded in gemma_after_sft_eval.ipynb and reported throughout
# gemma_final_report.ipynb).
EXPERIMENT_NAME = "v1"

DRIVE_BASE = "/content/drive/MyDrive/Colab Notebooks/Training_Data_Gemma"
OUTPUT_DIR = f"{DRIVE_BASE}/gemma3_1b_qlora_{EXPERIMENT_NAME}"

MODEL_NAME = "google/gemma-3-1b-it"
MAX_SEQ_LENGTH = 1536

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8

print("=" * 80)
print("EXPERIMENT CONFIGURATION")
print("=" * 80)

print("Experiment name:", EXPERIMENT_NAME)
print("Output directory:", OUTPUT_DIR)
print("Model:", MODEL_NAME)
print("Max sequence length:", MAX_SEQ_LENGTH)
print("LoRA r / alpha / dropout:", LORA_R, "/", LORA_ALPHA, "/", LORA_DROPOUT)
print("Epochs:", NUM_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print(
    "Effective batch size:",
    TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
)

print("\n\u2705 Experiment configuration is ready.")

In [ ]:
# This cell loads the Gemma 3 1B tokenizer, sets the padding token safely, and verifies that the chat template is available for training.

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("=" * 80)
print("GEMMA TOKENIZER CHECK")
print("=" * 80)

print("Model:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Pad token:", tokenizer.pad_token)
print("Pad token ID:", tokenizer.pad_token_id)
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)
print("Padding side:", tokenizer.padding_side)
print("Chat template available:", tokenizer.chat_template is not None)

if tokenizer.chat_template is None:
    raise RuntimeError("Gemma tokenizer does not have a chat template.")

print("\n✅ Gemma tokenizer is ready.")

In [ ]:
# This cell loads the teacher-generated GSM8K train and validation JSONL files into memory and inspects one example to verify the expected fields and message format.

import json

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_FILE)
val_rows = load_jsonl(VAL_FILE)

print("=" * 80)
print("DATASET LOADED")
print("=" * 80)

print("Train examples:", len(train_rows))
print("Validation examples:", len(val_rows))

sample = train_rows[0]

print("\nSample keys:")
print(sample.keys())

print("\nProblem ID:")
print(sample["problem_id"])

print("\nQuestion:")
print(sample["question"])

print("\nStudent target:")
print(sample["student_target"])

print("\nMessages:")
for message in sample["messages"]:
    print(message["role"], "->")
    print(message["content"])
    print()

print("✅ Dataset structure looks ready for tokenization.")

DATASET LOADED
Train examples: 1922
Validation examples: 485

Sample keys:
dict_keys(['problem_id', 'question', 'gold_answer', 'teacher_final_answer', 'student_target', 'messages'])

Problem ID:
gsm8k_train_1

Question:
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?

Student target:
<reasoning>
Step 1: Weng earns $12 per hour.  
Step 2: One hour has 60 minutes.  
Step 3: Weng worked 50 minutes.  
Step 4: To find the rate per minute, divide $12 by 60 minutes: 12 / 60 = 0.2 dollars per minute.  
Step 5: Multiply the rate per minute by the number of minutes worked: 0.2 * 50 = 10.
</reasoning>
<final_answer>10</final_answer>

Messages:
system ->
Solve the mathematical problem using step-by-step reasoning, with at most one arithmetic operation per step. Return the reasoning inside <reasoning> tags and the numerical answer inside <final_answer> tags.

user ->
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 

In [ ]:
# This cell converts the loaded JSONL rows into Hugging Face Dataset objects so they can be tokenized and passed into the Trainer.

from datasets import Dataset

train_dataset = Dataset.from_list(train_rows)
val_dataset = Dataset.from_list(val_rows)

print("=" * 80)
print("HUGGING FACE DATASET CHECK")
print("=" * 80)

print("Train dataset:", train_dataset)
print("Validation dataset:", val_dataset)

print("\nTrain rows:", len(train_dataset))
print("Validation rows:", len(val_dataset))

print("\nTrain columns:")
print(train_dataset.column_names)

print("\n✅ Hugging Face datasets are ready.")

HUGGING FACE DATASET CHECK
Train dataset: Dataset({
    features: ['problem_id', 'question', 'gold_answer', 'teacher_final_answer', 'student_target', 'messages'],
    num_rows: 1922
})
Validation dataset: Dataset({
    features: ['problem_id', 'question', 'gold_answer', 'teacher_final_answer', 'student_target', 'messages'],
    num_rows: 485
})

Train rows: 1922
Validation rows: 485

Train columns:
['problem_id', 'question', 'gold_answer', 'teacher_final_answer', 'student_target', 'messages']

✅ Hugging Face datasets are ready.


In [ ]:
# This cell tokenizes each example with Gemma's chat template and masks prompt tokens so training loss is computed only on the teacher assistant response.

def tokenize_example(example):
    messages = example["messages"]

    prompt_messages = messages[:2]

    prompt_ids = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
    )

    full_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
    )

    if len(full_ids) > MAX_SEQ_LENGTH:
        raise ValueError(
            f"Sequence length {len(full_ids)} exceeds MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}"
        )

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise ValueError("Prompt tokens do not match the beginning of full sequence.")

    labels = (
        [-100] * len(prompt_ids)
        + full_ids[len(prompt_ids):]
    )

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }

tokenized_train = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train set",
)

tokenized_val = val_dataset.map(
    tokenize_example,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation set",
)

print("=" * 80)
print("TOKENIZATION CHECK")
print("=" * 80)

print("Train examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_val))

train_lengths = [len(x["input_ids"]) for x in tokenized_train]
val_lengths = [len(x["input_ids"]) for x in tokenized_val]

print("Longest train sequence:", max(train_lengths))
print("Longest validation sequence:", max(val_lengths))
print("Max allowed sequence:", MAX_SEQ_LENGTH)

sample_labels = tokenized_train[0]["labels"]

ignored_tokens = sum(x == -100 for x in sample_labels)
supervised_tokens = sum(x != -100 for x in sample_labels)

print("\nFirst example:")
print("Prompt tokens masked:", ignored_tokens)
print("Assistant tokens learned:", supervised_tokens)
print("Total tokens:", len(sample_labels))

print("\n✅ Assistant-only tokenization is ready.")

In [ ]:
# This cell creates a dynamic padding collator that pads each batch only to its longest sequence and pads labels with -100 so padding does not affect training loss.

from dataclasses import dataclass
from typing import Any
import torch

@dataclass
class DynamicPaddingCollator:
    tokenizer: Any
    pad_to_multiple_of: int = 8

    def __call__(self, features):
        labels = [feature["labels"] for feature in features]

        model_features = [
            {
                "input_ids": feature["input_ids"],
                "attention_mask": feature["attention_mask"],
            }
            for feature in features
        ]

        batch = self.tokenizer.pad(
            model_features,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        max_length = batch["input_ids"].shape[1]

        padded_labels = []

        for label in labels:
            padding_length = max_length - len(label)

            padded_labels.append(
                label + [-100] * padding_length
            )

        batch["labels"] = torch.tensor(
            padded_labels,
            dtype=torch.long,
        )

        return batch


data_collator = DynamicPaddingCollator(tokenizer)

test_batch = data_collator(
    [
        tokenized_train[0],
        tokenized_train[1],
    ]
)

print("=" * 80)
print("DYNAMIC PADDING CHECK")
print("=" * 80)

print("input_ids shape:", test_batch["input_ids"].shape)
print("attention_mask shape:", test_batch["attention_mask"].shape)
print("labels shape:", test_batch["labels"].shape)

print(
    "Supervised tokens:",
    int((test_batch["labels"] != -100).sum())
)

print(
    "Ignored/padded label tokens:",
    int((test_batch["labels"] == -100).sum())
)

print("\n✅ Dynamic padding collator is ready.")

You're using a GemmaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


DYNAMIC PADDING CHECK
input_ids shape: torch.Size([2, 336])
attention_mask shape: torch.Size([2, 336])
labels shape: torch.Size([2, 336])
Supervised tokens: 370
Ignored/padded label tokens: 302

✅ Dynamic padding collator is ready.


In [ ]:
# This cell loads Gemma 3 1B in 4-bit NF4 with double quantization and FP16 compute so it is ready for memory-efficient QLoRA fine-tuning.

import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="eager",
)

model.config.use_cache = False

print("=" * 80)
print("GEMMA 3 1B LOADED FOR QLoRA")
print("=" * 80)

print("Model:", MODEL_NAME)
print("Loaded in 4-bit:", getattr(model, "is_loaded_in_4bit", False))
print("Quantization type: NF4")
print("Double quantization: True")
print("Compute dtype: FP16")
print("Attention implementation: eager")
print(
    "Model memory footprint:",
    round(model.get_memory_footprint() / (1024 ** 3), 3),
    "GiB",
)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated() / (1024 ** 3), 3),
        "GiB",
    )

print("\n✅ Gemma base model is ready for LoRA setup.")

In [ ]:
# This cell prepares the 4-bit Gemma model for QLoRA training, enables gradient checkpointing, and attaches LoRA adapters to the attention and MLP projection layers.

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# Prepare the quantized base model for training.
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

model.config.use_cache = False

# LoRA configuration, read from the experiment configuration above.
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(
    model,
    lora_config,
)

print("=" * 80)
print("QLoRA / LoRA CHECK")
print("=" * 80)

model.print_trainable_parameters()

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

total_params = sum(
    p.numel() for p in model.parameters()
)

print("\nTrainable parameters:", f"{trainable_params:,}")
print("Total parameters:", f"{total_params:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_params / total_params:.4f}%"
)

print("\n✅ Gemma is ready for QLoRA training.")

In [ ]:
# This cell builds the Hugging Face TrainingArguments from the experiment configuration defined above.

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    lr_scheduler_type="linear",
    warmup_ratio=0.03,

    weight_decay=0.01,
    max_grad_norm=1.0,

    optim="paged_adamw_8bit",

    fp16=True,
    bf16=False,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=10,
    save_total_limit=2,

    prediction_loss_only=True,

    seed=42,
    data_seed=42,

    report_to="none",
    remove_unused_columns=False,
)

print("=" * 80)
print("TRAINING CONFIGURATION")
print("=" * 80)

print("Output directory:", OUTPUT_DIR)
print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print(
    "Effective batch size:",
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)
print("Optimizer:", training_args.optim)
print("Scheduler:", training_args.lr_scheduler_type)
print("FP16:", training_args.fp16)
print("Evaluation:", training_args.eval_strategy)
print("Save strategy:", training_args.save_strategy)

print("\n\u2705 Training configuration is ready.")

In [ ]:
# This cell creates the Hugging Face Trainer using the QLoRA model, tokenized datasets, dynamic padding collator, and the tuned training configuration.

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("=" * 80)
print("TRAINER CHECK")
print("=" * 80)

print("Train examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))
print("Epochs:", trainer.args.num_train_epochs)
print("Learning rate:", trainer.args.learning_rate)
print("Gradient accumulation:", trainer.args.gradient_accumulation_steps)
print("Output directory:", trainer.args.output_dir)

print("\n✅ Trainer is ready.")

In [ ]:
# This cell starts Gemma QLoRA training and automatically resumes from the latest saved checkpoint if Colab disconnected during an earlier run.

import os
import time
import torch
from transformers.trainer_utils import get_last_checkpoint

print("=" * 80)
print("STARTING / RESUMING GEMMA 3 1B QLoRA TRAINING")
print("=" * 80)

# Look for an existing checkpoint in Google Drive.
last_checkpoint = None

if os.path.isdir(OUTPUT_DIR):
    last_checkpoint = get_last_checkpoint(OUTPUT_DIR)

if last_checkpoint is not None:
    print("✅ Existing checkpoint found:")
    print(last_checkpoint)
    print("\nTraining will resume from this checkpoint.")
else:
    print("No previous checkpoint found.")
    print("Training will start from the beginning.")

print("\nEpochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print(
    "Effective batch size:",
    training_args.per_device_train_batch_size
    * training_args.gradient_accumulation_steps
)
print("Train examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_val))
print("Output directory:", OUTPUT_DIR)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.time()

train_result = trainer.train(
    resume_from_checkpoint=last_checkpoint
)

elapsed_minutes = (time.time() - start_time) / 60

print("\n" + "=" * 80)
print("TRAINING COMPLETE")
print("=" * 80)

print(f"Training time this session: {elapsed_minutes:.2f} minutes")
print(f"Final training loss: {train_result.training_loss:.6f}")

if torch.cuda.is_available():
    print(
        "Peak GPU memory:",
        round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3),
        "GiB",
    )

print("\nBest checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)

print("\n✅ Training finished successfully.")

In [ ]:
# This cell saves the best LoRA adapter and tokenizer to Drive and writes a training_summary.json capturing the exact run configuration.

FINAL_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "final_best_adapter")

trainer.save_model(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

training_summary = {
    "experiment_name": EXPERIMENT_NAME,
    "base_model": MODEL_NAME,
    "training_method": "4-bit QLoRA",
    "quantization": "NF4",
    "compute_dtype": "float16",
    "train_examples": len(tokenized_train),
    "validation_examples": len(tokenized_val),
    "epochs": training_args.num_train_epochs,
    "per_device_train_batch_size": training_args.per_device_train_batch_size,
    "gradient_accumulation_steps": training_args.gradient_accumulation_steps,
    "effective_batch_size": (
        training_args.per_device_train_batch_size
        * training_args.gradient_accumulation_steps
    ),
    "learning_rate": training_args.learning_rate,
    "optimizer": str(training_args.optim),
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "lora_dropout": lora_config.lora_dropout,
    "trainable_parameters": trainable_params,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_eval_loss": trainer.state.best_metric,
    "elapsed_seconds_this_session": elapsed_minutes * 60,
    "trainer_train_runtime": train_result.metrics.get("train_runtime"),
    "final_adapter_path": FINAL_ADAPTER_DIR,
}

summary_path = os.path.join(FINAL_ADAPTER_DIR, "training_summary.json")

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(training_summary, f, indent=2)

print("=" * 80)
print("FINAL ADAPTER SAVED")
print("=" * 80)

print("Adapter directory:", FINAL_ADAPTER_DIR)
print("Training summary:", summary_path)

for key, value in training_summary.items():
    print(f"  {key}: {value}")

print()
print("\u2705 Final adapter and training summary saved.")